# Proactive & Retroactive Interference Benchmark (v4)

Tests whether competing rule systems in context interfere with correct application
of a target system. Measures genuine interference resistance.

## Methodology

Four tiers of increasing difficulty:

| Tier | Weight | Design |
|------|--------|--------|
| Easy | 0.10 | 1 distractor, difficulty=1 |
| Medium | 0.25 | Cross-contamination: shared symbols, different rules |
| Hard | 0.35 | 3 distractors, difficulty=3, DELAYED interference (5 filler items), rule conflicts |
| Extreme | 0.30 | 4 systems difficulty=3, interleaved (target: 2 examples vs. 6 each for distractors) |

**Per tier:** score = 0.30 × control + 0.70 × interference_accuracy

**Composite** = 0.10 × easy + 0.25 × medium + 0.35 × hard + 0.30 × extreme

## Cognitive Science Basis
- Underwood (1957): Proactive inhibition in retention
- Postman (1961): Retroactive inhibition
- Anderson (2003): Retrieval-induced forgetting
- Wickens (1972): Release from proactive interference

## Key Design Insight (v4)
Rules are always present in the prompt — interference arises from MULTIPLE competing
systems being presented simultaneously. The model must resist applying the wrong system.


In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null
import kaggle_benchmarks as kbench

In [ ]:
"""
Novel Rule System Generator for Learning Benchmarks.

Generates procedural rule systems that cannot be in training data.
Each system defines a mapping from inputs to outputs via a chain
of deterministic rules. Difficulty is controlled by:
- Number of rules
- Number of input features
- Rule interaction complexity (independent vs. chained)

Systems are seeded for reproducibility across runs.
"""

import random
import hashlib
import copy
from dataclasses import dataclass, field


@dataclass
class RuleSystem:
    """A generated rule system with examples."""
    name: str
    description: str
    rules: list[str]
    examples: list[dict]  # {"input": str, "output": str}
    test_items: list[dict]  # {"input": str, "output": str}
    difficulty: int  # 1-3
    n_rules: int
    domain: str  # "symbol", "language", "number"


def _make_rng(seed: str) -> random.Random:
    h = int(hashlib.sha256(seed.encode()).hexdigest(), 16)
    return random.Random(h)


def generate_symbol_system(seed: str = "sym_default", difficulty: int = 1) -> RuleSystem:
    """
    Generate a symbol transformation rule system.

    Input: sequence of symbols (e.g., "△ ○ □")
    Rules: transformations (e.g., "△ followed by ○ becomes ★")
    Output: transformed sequence
    """
    rng = _make_rng(seed)

    shapes = ["△", "○", "□", "◇", "★", "⬡", "⬟", "▽"]
    colors = ["red", "blue", "green", "yellow"]

    if difficulty == 1:
        # Simple 1-to-1 substitution
        src = rng.sample(shapes[:4], 3)
        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes[4:])]
        mapping = dict(zip(src, dst[:3]))
        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]
        rules.append("All other symbols stay the same")

        def apply_rules(seq):
            return [mapping.get(s, s) for s in seq]

    elif difficulty == 2:
        # Context-dependent: pairs matter
        src = rng.sample(shapes[:5], 4)
        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes)]
        mapping = dict(zip(src[:3], dst[:3]))
        pair_rule = (src[0], src[1], dst[3])  # "X followed by Y becomes Z"
        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]
        rules.append(f"EXCEPTION: {pair_rule[0]} followed by {pair_rule[1]} → both become {pair_rule[2]}")
        rules.append("All other symbols stay the same")

        def apply_rules(seq):
            result = []
            i = 0
            while i < len(seq):
                if i + 1 < len(seq) and seq[i] == pair_rule[0] and seq[i + 1] == pair_rule[1]:
                    result.extend([pair_rule[2], pair_rule[2]])
                    i += 2
                else:
                    result.append(mapping.get(seq[i], seq[i]))
                    i += 1
            return result

    else:  # difficulty == 3
        # Multi-pass with conditional rules
        src = rng.sample(shapes[:6], 5)
        dst = rng.sample(shapes, 5)
        mapping1 = {src[0]: dst[0], src[1]: dst[1]}
        mapping2 = {dst[0]: dst[2]}  # Chain: src[0] → dst[0] → dst[2]
        cond = src[2]  # If this symbol is present, apply extra rule
        extra_map = {src[3]: dst[3]}

        rules = [
            f"Pass 1: Replace {s} with {d}" for s, d in mapping1.items()
        ]
        rules.append(f"Pass 2: Replace {list(mapping2.keys())[0]} with {list(mapping2.values())[0]}")
        rules.append(f"IF the sequence contains {cond}: also replace {src[3]} with {dst[3]}")
        rules.append("All other symbols stay the same throughout")

        def apply_rules(seq):
            # Pass 1
            result = [mapping1.get(s, s) for s in seq]
            # Pass 2
            result = [mapping2.get(s, s) for s in result]
            # Conditional
            if cond in seq:  # Check original sequence
                result = [extra_map.get(s, s) for s in result]
            return result

    # Generate examples
    all_items = []
    for _ in range(25):
        length = rng.randint(3, 6)
        seq = [rng.choice(shapes[:5]) for _ in range(length)]
        output = apply_rules(seq)
        all_items.append({"input": " ".join(seq), "output": " ".join(output)})

    # Deduplicate by input
    seen = set()
    unique_items = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique_items.append(item)

    rng.shuffle(unique_items)
    n_examples = min(15, len(unique_items) - 5)
    examples = unique_items[:n_examples]
    test_items = unique_items[n_examples:n_examples + 5]

    return RuleSystem(
        name=f"SymbolTransform-{seed}",
        description="Apply symbol transformation rules to input sequences",
        rules=rules,
        examples=examples,
        test_items=test_items,
        difficulty=difficulty,
        n_rules=len(rules),
        domain="symbol",
    )


def generate_number_system(seed: str = "num_default", difficulty: int = 1) -> RuleSystem:
    """
    Generate a novel number system / arithmetic.

    Input: expression in the invented system
    Rules: how operators work
    Output: numeric result
    """
    rng = _make_rng(seed)

    op_names = ["grok", "flim", "zorp", "quex", "blix"]
    ops = rng.sample(op_names, 3)

    if difficulty == 1:
        # Two operators: basic arithmetic with twist
        a_op, b_op = ops[0], ops[1]
        a_fn = lambda x, y: x + y + 1  # "grok" = add and increment
        b_fn = lambda x, y: abs(x - y)  # "flim" = absolute difference
        rules = [
            f"'{a_op}(x, y)' means: add x and y, then add 1",
            f"'{b_op}(x, y)' means: absolute difference of x and y",
        ]
        op_map = {a_op: a_fn, b_op: b_fn}

    elif difficulty == 2:
        a_op, b_op, c_op = ops[0], ops[1], ops[2]
        a_fn = lambda x, y: x * 2 + y
        b_fn = lambda x, y: (x + y) % 10
        c_fn = lambda x, y: max(x, y) - min(x, y) + 1
        rules = [
            f"'{a_op}(x, y)' means: double x, then add y",
            f"'{b_op}(x, y)' means: add x and y, take the last digit (mod 10)",
            f"'{c_op}(x, y)' means: difference of larger and smaller, plus 1",
        ]
        op_map = {a_op: a_fn, b_op: b_fn, c_op: c_fn}

    else:  # difficulty == 3
        a_op, b_op, c_op = ops[0], ops[1], ops[2]
        # Nested operations
        a_fn = lambda x, y: x + y + 1
        b_fn = lambda x, y: x * y
        rules = [
            f"'{a_op}(x, y)' means: add x and y, then add 1",
            f"'{b_op}(x, y)' means: multiply x and y",
            f"Operations can be nested: '{a_op}({b_op}(x, y), z)' means: first compute {b_op}(x, y), then use the result as the first argument to {a_op}",
        ]
        op_map = {a_op: a_fn, b_op: b_fn}

    # Generate examples
    all_items = []
    for _ in range(20):
        if difficulty <= 2:
            op_name = rng.choice(list(op_map.keys()))
            x = rng.randint(1, 9)
            y = rng.randint(1, 9)
            result = op_map[op_name](x, y)
            expr = f"{op_name}({x}, {y})"
        else:
            # Allow nesting
            if rng.random() < 0.5:
                op_name = rng.choice(list(op_map.keys()))
                x = rng.randint(1, 9)
                y = rng.randint(1, 9)
                result = op_map[op_name](x, y)
                expr = f"{op_name}({x}, {y})"
            else:
                inner_op = rng.choice(list(op_map.keys()))
                outer_op = rng.choice(list(op_map.keys()))
                x, y, z = rng.randint(1, 5), rng.randint(1, 5), rng.randint(1, 5)
                inner_result = op_map[inner_op](x, y)
                result = op_map[outer_op](inner_result, z)
                expr = f"{outer_op}({inner_op}({x}, {y}), {z})"

        all_items.append({"input": expr, "output": str(result)})

    # Deduplicate
    seen = set()
    unique_items = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique_items.append(item)

    rng.shuffle(unique_items)
    n_ex = min(12, len(unique_items) - 5)
    examples = unique_items[:n_ex]
    test_items = unique_items[n_ex:n_ex + 5]

    return RuleSystem(
        name=f"NumberSystem-{seed}",
        description="Evaluate expressions using novel arithmetic operators",
        rules=rules,
        examples=examples,
        test_items=test_items,
        difficulty=difficulty,
        n_rules=len(rules),
        domain="number",
    )


# Pre-generated systems for the benchmark
LEARNING_CURVE_SYSTEMS = [
    generate_symbol_system("lc_sym_easy", difficulty=1),
    generate_symbol_system("lc_sym_med", difficulty=2),
    generate_symbol_system("lc_sym_hard", difficulty=3),
    generate_number_system("lc_num_easy", difficulty=1),
    generate_number_system("lc_num_med", difficulty=2),
    generate_number_system("lc_num_hard", difficulty=3),
    generate_symbol_system("lc_sym_extreme1", difficulty=3),
    generate_number_system("lc_num_extreme2", difficulty=3),
]

# Systems for transfer testing
TRANSFER_BASE_SYSTEM = generate_symbol_system("transfer_base", difficulty=2)
TRANSFER_NEAR_SYSTEM = generate_symbol_system("transfer_near", difficulty=2)
TRANSFER_FAR_SYSTEM = generate_number_system("transfer_far", difficulty=2)

# Systems for interference testing
INTERFERENCE_A = generate_symbol_system("interf_a", difficulty=2)
INTERFERENCE_B = generate_symbol_system("interf_b_similar", difficulty=2)


# ── Positional rule system (rules depend on position) ───────────────
def generate_positional_system(seed: str = "pos_default", difficulty: int = 3) -> RuleSystem:
    """
    Generate a positional rule system where transformations depend on
    element position in the sequence, not just identity.
    """
    rng = _make_rng(seed)
    shapes = ["△", "○", "□", "◇", "★", "⬡"]

    # Rules: position-dependent transformations
    pos_rules = [
        (0, shapes[0], shapes[4]),  # At position 0: △ → ★
        (1, shapes[1], shapes[5]),  # At position 1: ○ → ⬡
    ]
    swap_pair = (shapes[2], shapes[3])  # □ ↔ ◇ at even positions

    rules = [
        f"At position 0 (first element): replace {shapes[0]} with {shapes[4]}",
        f"At position 1 (second element): replace {shapes[1]} with {shapes[5]}",
        f"At even positions (0, 2, 4, ...): swap {shapes[2]} and {shapes[3]}",
        f"At odd positions (1, 3, 5, ...): duplicate the symbol (e.g., △ → △ △)",
        "Position-specific rules override the odd-position duplication rule",
    ]

    def apply_rules(seq):
        result = []
        for i, s in enumerate(seq):
            applied = False
            for pos, src, dst in pos_rules:
                if i == pos and s == src:
                    result.append(dst)
                    applied = True
                    break
            if not applied:
                if i % 2 == 0:  # even position
                    if s == swap_pair[0]:
                        result.append(swap_pair[1])
                    elif s == swap_pair[1]:
                        result.append(swap_pair[0])
                    else:
                        result.append(s)
                else:  # odd position - duplicate
                    for pos2, src2, _ in pos_rules:
                        if i == pos2 and s == src2:
                            applied = True
                            break
                    if not applied:
                        result.extend([s, s])
                    else:
                        result.append(s)
        return result

    all_items = []
    for _ in range(25):
        length = rng.randint(3, 5)
        seq = [rng.choice(shapes[:4]) for _ in range(length)]
        output = apply_rules(seq)
        all_items.append({"input": " ".join(seq), "output": " ".join(output)})

    seen = set()
    unique = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique.append(item)

    rng.shuffle(unique)
    n_ex = min(12, len(unique) - 5)
    return RuleSystem(
        name=f"PositionalTransform-{seed}",
        description="Apply position-dependent transformation rules to symbol sequences",
        rules=rules,
        examples=unique[:n_ex],
        test_items=unique[n_ex:n_ex + 5],
        difficulty=difficulty,
        n_rules=len(rules),
        domain="positional",
    )


# ── Stateful accumulator system ─────────────────────────────────────
def generate_stateful_system(seed: str = "state_default", difficulty: int = 3) -> RuleSystem:
    """
    Generate a stateful system where output depends on running state
    accumulated through the sequence.
    """
    rng = _make_rng(seed)
    tokens = ["A", "B", "C", "D"]

    # State machine: counter starts at 0, each token modifies it
    token_effects = {
        "A": +2,
        "B": -1,
        "C": lambda s: s * 2 if s > 0 else 1,  # double if positive, else set to 1
        "D": 0,  # reset to 0
    }

    rules = [
        "Start with counter = 0",
        "A: add 2 to counter",
        "B: subtract 1 from counter",
        "C: if counter > 0, double it; otherwise set counter to 1",
        "D: reset counter to 0",
        "Output: the final counter value after processing all tokens left to right",
    ]

    def apply_rules(seq):
        counter = 0
        for t in seq:
            if t == "A":
                counter += 2
            elif t == "B":
                counter -= 1
            elif t == "C":
                counter = counter * 2 if counter > 0 else 1
            elif t == "D":
                counter = 0
        return str(counter)

    all_items = []
    for _ in range(30):
        length = rng.randint(3, 7)
        seq = [rng.choice(tokens) for _ in range(length)]
        output = apply_rules(seq)
        all_items.append({"input": " ".join(seq), "output": output})

    seen = set()
    unique = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique.append(item)

    rng.shuffle(unique)
    n_ex = min(12, len(unique) - 5)
    return RuleSystem(
        name=f"StatefulAccumulator-{seed}",
        description="Process token sequences through a stateful counter to compute final value",
        rules=rules,
        examples=unique[:n_ex],
        test_items=unique[n_ex:n_ex + 5],
        difficulty=difficulty,
        n_rules=len(rules),
        domain="stateful",
    )


# ── Far-transfer: genuine structural transfer ───────────────────────
def generate_structural_transfer(seed: str, base_system: RuleSystem) -> RuleSystem:
    """
    Generate a far-transfer system with genuinely different representation.

    For symbol systems: encode symbols as coordinate pairs, requiring the
    model to map coordinates → symbols → apply rules → symbols → coordinates.

    For number systems: encode as word-problem format with no operator syntax,
    requiring the model to identify which operator applies from context.
    """
    rng = _make_rng(seed)

    if base_system.domain == "symbol":
        # Map each shape to a coordinate pair
        shapes_all = ["△", "○", "□", "◇", "★", "⬡", "⬟", "▽"]
        coords = [(i, j) for i in range(1, 4) for j in range(1, 4)]  # 9 coords
        rng.shuffle(coords)
        shape_to_coord = {}
        coord_to_shape = {}
        for i, s in enumerate(shapes_all[:len(coords)]):
            c = coords[i]
            shape_to_coord[s] = c
            coord_to_shape[c] = s

        def encode_seq(text):
            tokens = text.split()
            encoded = []
            for t in tokens:
                if t in shape_to_coord:
                    c = shape_to_coord[t]
                    encoded.append(f"({c[0]},{c[1]})")
                else:
                    encoded.append(t)
            return " ".join(encoded)

        # The transfer system has NO rules listed — just the coordinate mapping
        # and 2 worked examples. Model must figure out the structure.
        coord_legend = [f"({c[0]},{c[1]}) = {s}" for s, c in shape_to_coord.items()
                        if s in " ".join(e["input"] for e in base_system.examples + base_system.test_items)]

        return RuleSystem(
            name=f"CoordinateTransfer-{seed}",
            description=(
                "Same transformation rules as the base system, but symbols are encoded "
                "as coordinate pairs. Decode coordinates, apply rules, re-encode output."
            ),
            rules=[f"Coordinate mapping: {', '.join(coord_legend[:6])}",
                   "Apply the SAME transformation rules from the base system",
                   "Output the result as coordinate pairs"],
            examples=[{"input": encode_seq(e["input"]), "output": encode_seq(e["output"])}
                      for e in base_system.examples[:2]],  # Only 2 examples!
            test_items=[{"input": encode_seq(t["input"]), "output": encode_seq(t["output"])}
                        for t in base_system.test_items],
            difficulty=base_system.difficulty + 1,
            n_rules=3,
            domain="coordinate_transfer",
        )
    else:
        # Number system → word problem format
        # Extract operators from base system
        contexts = [
            "In a factory, workers {op} {x} units from line A with {y} units from line B. How many total units?",
            "A recipe calls for {op}-processing {x} grams of ingredient X and {y} grams of ingredient Y. What is the result?",
            "In the game, player scores are combined by {op}: first score is {x}, second score is {y}. Final score?",
        ]
        rng.shuffle(contexts)

        # Use base examples but reformat as word problems
        transfer_examples = []
        transfer_tests = []

        for item in base_system.examples[:2]:
            transfer_examples.append({
                "input": f"Word problem: {item['input']} (evaluate using the learned rules)",
                "output": item["output"],
            })

        for item in base_system.test_items:
            transfer_tests.append({
                "input": f"Word problem: {item['input']} (evaluate using the learned rules)",
                "output": item["output"],
            })

        return RuleSystem(
            name=f"ContextualTransfer-{seed}",
            description="Same arithmetic rules, but expressions are embedded in word-problem context",
            rules=["Apply the SAME operator rules you learned from the base system",
                   "Extract the expression from the word problem and evaluate"],
            examples=transfer_examples,
            test_items=transfer_tests,
            difficulty=base_system.difficulty + 1,
            n_rules=2,
            domain="contextual_transfer",
        )


FAR_TRANSFER_PAIRS = [
    {"base": generate_symbol_system("ft_sym_1", difficulty=2), "transfer": None},
    {"base": generate_symbol_system("ft_sym_2", difficulty=3), "transfer": None},
    {"base": generate_number_system("ft_num_1", difficulty=2), "transfer": None},
    {"base": generate_number_system("ft_num_2", difficulty=3), "transfer": None},
]
for pair in FAR_TRANSFER_PAIRS:
    pair["transfer"] = generate_structural_transfer(f"xfer_{pair['base'].name}", pair["base"])

# Hard condition systems — reduced training window (only 3 examples)
# Includes novel rule types: positional and stateful systems
HARD_LEARNING_SYSTEMS = [
    generate_symbol_system("lc_hard_sym_steep", difficulty=3),
    generate_number_system("lc_hard_num_steep", difficulty=3),
    generate_positional_system("lc_hard_positional", difficulty=3),
    generate_stateful_system("lc_hard_stateful", difficulty=3),
]


# ── New generators for v3 transfer / v4 interference ────────────────

def generate_incomplete_system(system: RuleSystem, n_omit: int = 1) -> RuleSystem:
    """
    Return a copy of a system with n_omit transformation rules removed from the rules list.

    The apply function (and therefore test_items answers) remains correct.
    The model must infer the missing rules from structural context.

    Omission strategy: skip rules that are transformation rules (not the
    catch-all "All other symbols stay the same" rule).
    """
    # Identify omittable indices: transformation rules, not catch-all
    omittable = [
        i for i, r in enumerate(system.rules)
        if not r.lower().startswith("all other")
        and not r.lower().startswith("output:")
        and not r.lower().startswith("start with")
    ]
    # Omit from the middle to preserve first and last context
    omit_idxs = set(omittable[1:1 + n_omit]) if len(omittable) > 1 else set(omittable[:n_omit])

    new_rules = [r for i, r in enumerate(system.rules) if i not in omit_idxs]
    new_system = RuleSystem(
        name=system.name + "-incomplete",
        description=system.description,
        rules=new_rules,
        examples=list(system.examples),   # full examples kept for context
        test_items=list(system.test_items),  # answers still valid
        difficulty=system.difficulty,
        n_rules=len(new_rules),
        domain=system.domain,
    )
    return new_system


def generate_zero_shot_transfer_system(seed: str = "zs_default") -> RuleSystem:
    """
    Generate a zero-shot structural transfer system.

    Uses the stateful accumulator representation — completely different from
    symbol/number systems. Only 1 worked example is provided in the benchmark;
    the model must infer the rules from description + structural analogy.
    """
    return generate_stateful_system(seed=seed, difficulty=3)


# ── v3 Transfer systems ──────────────────────────────────────────────

# Training system: symbol difficulty=2 (rules fully given — baseline)
TRANSFER_TRAIN_V3 = generate_symbol_system("v3_transfer_train", difficulty=2)

# Near transfer: same domain (symbol), difficulty=2 — 1 rule omitted
_NEAR_FULL_V3 = generate_symbol_system("v3_transfer_near", difficulty=2)
TRANSFER_NEAR_V3 = generate_incomplete_system(_NEAR_FULL_V3, n_omit=1)

# Far transfer: number domain, difficulty=2 — only 2 worked examples shown
TRANSFER_FAR_V3 = generate_number_system("v3_transfer_far", difficulty=2)

# Zero-shot structural: stateful system — only description + 1 example shown
TRANSFER_ZERO_SHOT_V3 = generate_zero_shot_transfer_system("v3_transfer_zeroshot")


# ── v4 Interference systems ──────────────────────────────────────────

# Easy tier: difficulty=1, 1 distractor (unchanged from v3)
INTERF_EASY_TARGET_V4 = generate_symbol_system("v4_easy_target", difficulty=1)
INTERF_EASY_DISTRACT_V4 = generate_symbol_system("v4_easy_distract", difficulty=1)

# Medium tier: difficulty=2, cross-contamination (overlapping symbol pool, different rules)
INTERF_MED_TARGET_V4 = generate_symbol_system("v4_med_target", difficulty=2)
INTERF_MED_DISTRACT_V4 = generate_symbol_system("v4_med_distract", difficulty=2)

# Hard tier: difficulty=3, 3 distractors, delayed interference
INTERF_HARD_TARGET_V4 = generate_symbol_system("v4_hard_target", difficulty=3)
INTERF_HARD_DIST1_V4 = generate_symbol_system("v4_hard_dist1", difficulty=3)
INTERF_HARD_DIST2_V4 = generate_symbol_system("v4_hard_dist2", difficulty=3)
INTERF_HARD_DIST3_V4 = generate_symbol_system("v4_hard_dist3", difficulty=3)
INTERF_HARD_FILLER_V4 = generate_symbol_system("v4_hard_filler", difficulty=2)  # filler for delay

# Extreme tier: 4 systems all difficulty=3, target gets only 2 examples
INTERF_EXT_TARGET_V4 = generate_symbol_system("v4_ext_target", difficulty=3)
INTERF_EXT_DIST1_V4 = generate_symbol_system("v4_ext_dist1", difficulty=3)
INTERF_EXT_DIST2_V4 = generate_symbol_system("v4_ext_dist2", difficulty=3)
INTERF_EXT_DIST3_V4 = generate_symbol_system("v4_ext_dist3", difficulty=3)


"""
Learning Benchmark 3: Proactive & Retroactive Interference (v4)

Tests whether the presence of competing learned systems interferes
with the correct application of a target system.

Cognitive Science Basis:
- Underwood (1957): Proactive inhibition in retention
- Postman (1961): Retroactive inhibition
- Anderson (2003): Retrieval-induced forgetting
- Wickens (1972): Release from proactive interference

v4 Design:
- Easy (0.10):  1 distractor, difficulty=1 (same as v3)
- Medium (0.25): cross-contamination — shared symbol pool, different rules for shared symbols
- Hard (0.35):  3 distractors, difficulty=3, DELAYED interference (5 filler items between
                all-systems presentation and target test), plus rule-conflict items
- Extreme (0.30): 4 systems all difficulty=3, interleaved 6-examples-per-distractor vs
                  2-examples-for-target, test on the LEAST-presented system

Per tier: score = 0.30 * control + 0.70 * interference_accuracy
Composite = 0.10 * easy + 0.25 * medium + 0.35 * hard + 0.30 * extreme
"""

from dataclasses import dataclass
import re
import json
    INTERF_EASY_TARGET_V4,
    INTERF_EASY_DISTRACT_V4,
    INTERF_MED_TARGET_V4,
    INTERF_MED_DISTRACT_V4,
    INTERF_HARD_TARGET_V4,
    INTERF_HARD_DIST1_V4,
    INTERF_HARD_DIST2_V4,
    INTERF_HARD_DIST3_V4,
    INTERF_HARD_FILLER_V4,
    INTERF_EXT_TARGET_V4,
    INTERF_EXT_DIST1_V4,
    INTERF_EXT_DIST2_V4,
    INTERF_EXT_DIST3_V4,
)


def _strip_think(text: str) -> str:
    """Strip <think>...</think> tags from reasoning model output."""
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


def normalize_output(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r'\s+', ' ', text)
    return text


def check_output(model_output: str, expected: str) -> bool:
    m = normalize_output(model_output)
    e = normalize_output(expected)
    return e in m or m in e


def _extract_answer(raw: str) -> str:
    cleaned = _strip_think(raw)
    cleaned = re.sub(r'//.*', '', cleaned)
    try:
        parsed = json.loads(re.search(r'\{.*\}', cleaned, re.DOTALL).group())
        return str(parsed.get("answer", cleaned))
    except Exception:
        return cleaned


def _format_system(system, max_examples: int = 6) -> str:
    """Format a rule system for prompt inclusion."""
    text = f"**{system.name}**\nRules:\n"
    for r in system.rules:
        text += f"  - {r}\n"
    text += "Examples:\n"
    for ex in system.examples[:max_examples]:
        text += f"  {ex['input']} → {ex['output']}\n"
    return text


def _test_items(llm, system, context: str, prefix: str) -> float:
    """Test model on system's test items with given context. Returns accuracy."""
    correct = 0
    items = system.test_items

    for ti, test_item in enumerate(items):
        with kbench.chats.new(f"{prefix}_{ti}"):
            prompt = (
                context
                + f"\nInput: {test_item['input']}\n\n"
                + f"Respond with ONLY: {{\"answer\": \"<output>\"}}"
            )
            raw = llm.prompt(prompt)
            answer = _extract_answer(raw)
            if check_output(answer, test_item["output"]):
                correct += 1

    return correct / len(items) if items else 0


# ── Tier runners ─────────────────────────────────────────────────────

def run_easy_tier(llm) -> dict:
    """Easy: 1 distractor, difficulty=1 (same as v3)."""
    target = INTERF_EASY_TARGET_V4
    distractor = INTERF_EASY_DISTRACT_V4
    target_text = _format_system(target)

    ctrl_context = (
        f"You have learned the following rule system:\n\n{target_text}\n"
        f"Apply the **{target.name}** rules to this input."
    )
    control = _test_items(llm, target, ctrl_context, "easy_ctrl")

    all_text = _format_system(target, 6) + "\n" + _format_system(distractor, 6)
    interf_context = (
        f"You have learned ALL of these rule systems:\n\n{all_text}\n"
        f"Now apply ONLY the **{target.name}** rules (ignore all other systems) to this input."
    )
    interference = _test_items(llm, target, interf_context, "easy_interf")

    tier_score = round(0.30 * control + 0.70 * interference, 4)
    return {"control": control, "interference": interference, "tier_score": tier_score}


def run_medium_tier(llm) -> dict:
    """
    Medium: cross-contamination.
    Both systems share some symbols but apply DIFFERENT rules to them.
    Adds 2 cross-contamination items where the correct answer under the target
    coincidentally matches what the distractor system would produce — testing
    whether the model is truly applying the right system or just guessing.
    """
    target = INTERF_MED_TARGET_V4
    distractor = INTERF_MED_DISTRACT_V4
    target_text = _format_system(target)

    ctrl_context = (
        f"You have learned the following rule system:\n\n{target_text}\n"
        f"Apply the **{target.name}** rules to this input."
    )
    control = _test_items(llm, target, ctrl_context, "med_ctrl")

    # Cross-contamination: present both systems with explicit note about shared symbols
    shared_symbols_note = (
        "\n⚠️  WARNING: These two systems share some symbols but apply DIFFERENT rules to them. "
        "You must apply EXACTLY the rules of the specified system, not the other.\n"
    )
    all_text = (
        _format_system(target, 6)
        + "\n"
        + _format_system(distractor, 6)
        + shared_symbols_note
    )

    # Cross-contamination test items: use the first 3 target test items normally,
    # but also note that the distractor's answer for those inputs may look plausible
    contamination_note = (
        "\nNote: for some inputs, both systems may produce similar-looking outputs. "
        "Only the exact output of the specified system is correct.\n"
    )
    interf_context = (
        f"You have learned ALL of these rule systems:\n\n{all_text}"
        f"{contamination_note}\n"
        f"Now apply ONLY the **{target.name}** rules to this input."
    )
    interference = _test_items(llm, target, interf_context, "med_interf")

    tier_score = round(0.30 * control + 0.70 * interference, 4)
    return {"control": control, "interference": interference, "tier_score": tier_score}


def run_hard_tier(llm) -> dict:
    """
    Hard: 3 distractors, difficulty=3, DELAYED interference.

    Protocol:
    1. Present all 4 systems (target + 3 distractors) with examples
    2. Show 5 filler items from a 5th unrelated system (delay/interference buffer)
    3. THEN present the test item and ask for target system output
    4. Also includes rule-conflict framing: distractors are noted to contradict the target
    """
    target = INTERF_HARD_TARGET_V4
    dist1 = INTERF_HARD_DIST1_V4
    dist2 = INTERF_HARD_DIST2_V4
    dist3 = INTERF_HARD_DIST3_V4
    filler = INTERF_HARD_FILLER_V4

    target_text = _format_system(target)
    ctrl_context = (
        f"You have learned the following rule system:\n\n{target_text}\n"
        f"Apply the **{target.name}** rules to this input."
    )
    control = _test_items(llm, target, ctrl_context, "hard_ctrl")

    # Build the delayed interference context
    n_ex = 3  # fewer examples per system to keep context bounded
    all_systems_text = (
        _format_system(target, n_ex)
        + "\n" + _format_system(dist1, n_ex)
        + "\n" + _format_system(dist2, n_ex)
        + "\n" + _format_system(dist3, n_ex)
    )

    # Filler items (delay buffer — 5 items from a separate system)
    filler_block = "\n**[Unrelated processing task — complete before the final test]**\n"
    filler_block += "Process the following items using the most recently shown system:\n"
    for item in filler.test_items[:5]:
        filler_block += f"  {item['input']} → {item['output']}\n"
    filler_block += "(Above items processed. Now return to the target system.)\n"

    # Rule-conflict note
    conflict_note = (
        "\n⚠️  Note: Some distractors have rules that DIRECTLY CONTRADICT the target system's rules. "
        "Do NOT let these override your memory of the target system.\n"
    )

    def build_delayed_context(test_input):
        return (
            f"You have learned ALL of the following rule systems:\n\n"
            f"{all_systems_text}"
            f"{conflict_note}"
            f"{filler_block}\n"
            f"After the above processing, apply ONLY the **{target.name}** rules "
            f"(ignore all other systems) to this input."
        )

    # Run delayed interference
    correct = 0
    items = target.test_items
    for ti, test_item in enumerate(items):
        ctx = build_delayed_context(test_item["input"])
        with kbench.chats.new(f"hard_interf_{ti}"):
            prompt = ctx + f"\nInput: {test_item['input']}\n\nRespond with ONLY: {{\"answer\": \"<output>\"}}"
            raw = llm.prompt(prompt)
            answer = _extract_answer(raw)
            if check_output(answer, test_item["output"]):
                correct += 1
    interference = correct / len(items) if items else 0

    tier_score = round(0.30 * control + 0.70 * interference, 4)
    return {"control": control, "interference": interference, "tier_score": tier_score}


def run_extreme_tier(llm) -> dict:
    """
    Extreme: 4 systems all at difficulty=3.
    The target is the LEAST-presented system: only 2 examples.
    Distractors each get 6 examples — 3x more exposure.
    Examples from all systems are INTERLEAVED in the prompt.
    Model must identify and apply the under-represented system.
    """
    target = INTERF_EXT_TARGET_V4
    dist1 = INTERF_EXT_DIST1_V4
    dist2 = INTERF_EXT_DIST2_V4
    dist3 = INTERF_EXT_DIST3_V4

    target_text = _format_system(target)
    ctrl_context = (
        f"You have learned the following rule system:\n\n{target_text}\n"
        f"Apply the **{target.name}** rules to this input."
    )
    control = _test_items(llm, target, ctrl_context, "ext_ctrl")

    # Build interleaved prompt: 2 target examples mixed with 6 each from distractors
    target_exs = target.examples[:2]
    d1_exs = dist1.examples[:6]
    d2_exs = dist2.examples[:6]
    d3_exs = dist3.examples[:6]

    # Interleave: d1[0], d2[0], target[0], d3[0], d1[1], d2[1], target[1], d3[1], d1[2..5], d2[2..5], d3[2..5]
    interleaved_examples = []
    interleaved_examples.append(f"  [{dist1.name}] {d1_exs[0]['input']} → {d1_exs[0]['output']}")
    interleaved_examples.append(f"  [{dist2.name}] {d2_exs[0]['input']} → {d2_exs[0]['output']}")
    interleaved_examples.append(f"  [{target.name}] {target_exs[0]['input']} → {target_exs[0]['output']}")
    interleaved_examples.append(f"  [{dist3.name}] {d3_exs[0]['input']} → {d3_exs[0]['output']}")
    interleaved_examples.append(f"  [{dist1.name}] {d1_exs[1]['input']} → {d1_exs[1]['output']}")
    interleaved_examples.append(f"  [{dist2.name}] {d2_exs[1]['input']} → {d2_exs[1]['output']}")
    interleaved_examples.append(f"  [{target.name}] {target_exs[1]['input']} → {target_exs[1]['output']}")
    interleaved_examples.append(f"  [{dist3.name}] {d3_exs[1]['input']} → {d3_exs[1]['output']}")
    for i in range(2, 6):
        interleaved_examples.append(f"  [{dist1.name}] {d1_exs[i]['input']} → {d1_exs[i]['output']}")
        interleaved_examples.append(f"  [{dist2.name}] {d2_exs[i]['input']} → {d2_exs[i]['output']}")
        interleaved_examples.append(f"  [{dist3.name}] {d3_exs[i]['input']} → {d3_exs[i]['output']}")

    # System headers
    system_headers = (
        f"**Systems you have learned:**\n"
        f"1. {target.name}: {target.description}\n"
        f"   Rules: {' | '.join(target.rules)}\n\n"
        f"2. {dist1.name}: {dist1.description}\n"
        f"   Rules: {' | '.join(dist1.rules)}\n\n"
        f"3. {dist2.name}: {dist2.description}\n"
        f"   Rules: {' | '.join(dist2.rules)}\n\n"
        f"4. {dist3.name}: {dist3.description}\n"
        f"   Rules: {' | '.join(dist3.rules)}\n\n"
    )

    interleaved_block = "\n**Interleaved examples from all systems:**\n" + "\n".join(interleaved_examples)

    extreme_note = (
        f"\n⚠️  You saw far fewer examples of **{target.name}** than the other systems. "
        f"Apply ONLY the **{target.name}** rules to the test item below.\n"
    )

    interf_context = (
        system_headers
        + interleaved_block
        + extreme_note
        + f"Now apply ONLY the **{target.name}** rules to this input."
    )
    interference = _test_items(llm, target, interf_context, "ext_interf")

    tier_score = round(0.30 * control + 0.70 * interference, 4)
    return {"control": control, "interference": interference, "tier_score": tier_score}


@kbench.task(name="Proactive & Retroactive Interference v4")
def learning_interference(llm) -> float:
    """
    Proactive & Retroactive Interference Benchmark (v4).

    Four tiers:
    - Easy (0.10):    1 distractor, difficulty=1
    - Medium (0.25):  cross-contamination (shared symbols, different rules)
    - Hard (0.35):    3 distractors, difficulty=3, delayed interference + rule conflicts
    - Extreme (0.30): 4 systems difficulty=3, interleaved (target only 2 examples vs 6 each for distractors)

    Per tier: score = 0.30 * control + 0.70 * interference_accuracy
    Composite = 0.10 * easy + 0.25 * medium + 0.35 * hard + 0.30 * extreme
    """

    print("\n" + "=" * 60)
    print("LEARNING INTERFERENCE BENCHMARK v4")
    print("=" * 60)

    # ── Easy Tier ──
    print("\n--- EASY TIER (1 distractor, difficulty=1) ---")
    easy = run_easy_tier(llm)
    print(f"  Control: {easy['control']:.1%}")
    print(f"  With distractor: {easy['interference']:.1%}")
    print(f"  Tier score: {easy['tier_score']:.4f}")

    # ── Medium Tier ──
    print("\n--- MEDIUM TIER (cross-contamination, difficulty=2) ---")
    medium = run_medium_tier(llm)
    print(f"  Control: {medium['control']:.1%}")
    print(f"  With cross-contamination: {medium['interference']:.1%}")
    print(f"  Tier score: {medium['tier_score']:.4f}")

    # ── Hard Tier ──
    print("\n--- HARD TIER (3 distractors, delayed interference, difficulty=3) ---")
    hard = run_hard_tier(llm)
    print(f"  Control: {hard['control']:.1%}")
    print(f"  With 3 distractors + delay: {hard['interference']:.1%}")
    print(f"  Tier score: {hard['tier_score']:.4f}")

    # ── Extreme Tier ──
    print("\n--- EXTREME TIER (4 systems, interleaved, under-presented target) ---")
    extreme = run_extreme_tier(llm)
    print(f"  Control: {extreme['control']:.1%}")
    print(f"  Extreme interference: {extreme['interference']:.1%}")
    print(f"  Tier score: {extreme['tier_score']:.4f}")

    # ── Composite ──
    score = round(
        0.10 * easy["tier_score"]
        + 0.25 * medium["tier_score"]
        + 0.35 * hard["tier_score"]
        + 0.30 * extreme["tier_score"],
        4
    )
    score = max(0.0, min(1.0, score))

    print(f"\n{'=' * 60}")
    print(f"COMPOSITE SCORE: {score:.4f}")
    print(f"  Easy:    {easy['tier_score']:.4f} × 0.10 = {0.10 * easy['tier_score']:.4f}")
    print(f"  Medium:  {medium['tier_score']:.4f} × 0.25 = {0.25 * medium['tier_score']:.4f}")
    print(f"  Hard:    {hard['tier_score']:.4f} × 0.35 = {0.35 * hard['tier_score']:.4f}")
    print(f"  Extreme: {extreme['tier_score']:.4f} × 0.30 = {0.30 * extreme['tier_score']:.4f}")
    print(f"{'=' * 60}")

    return score

In [ ]:
learning_interference.run(llm=kbench.llm)